# Notebook 1: Extracción y Preprocesamiento - Dra. Gabriela Jufe (Psicofarmacología Práctica)

Este notebook implementa el pipeline de ingesta, extracción estructurada, limpieza y normalización para el manual de referencia clínica **"Psicofarmacología Práctica" de la Dra. Gabriela Jufe**.

### Objetivos:
1. Extraer secciones textuales clínicas (principios activos, familias psicofarmacológicas, mecanismos, efectos adversos e interacciones).
2. Limpiar texto libre, corregir saltos de línea y errores típicos de formato PDF.
3. Descomponer efectos adversos e interacciones en formatos tabulares normalizados.
4. Mapear preliminarmente los principios activos a su código ATC de la OMS.
5. Exportar datasets limpios listos para la base de datos relacional SQLite.


In [2]:
%pip install pandas pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import sys
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

# Detectar la ruta raíz del proyecto de forma dinámica y robusta
current_dir = Path.cwd()
project_root = current_dir if (current_dir / 'data').exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_RAW = project_root / 'data' / 'raw'
DATA_PROCESSED = project_root / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

from src.extractors.jufe_extractor import JufeExtractor
from src.extractors.pdf_extractor import CachedPDFExtractor
from src.enrichers.atc_enricher import ATCEnricher

print(f'Entorno configurado correctamente. Raíz del proyecto: {project_root}')


Entorno configurado correctamente. Raíz del proyecto: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia


## 1. Función de Carga y Lectura (PDF o Texto)

El libro suele estar disponible como PDF o exportado a texto. Implementamos un lector flexible que intenta leer el PDF desde `data/raw/jufe_psicofarmacologia.pdf`, y en caso de no estar presente, carga un conjunto de datos representativo extraído de los capítulos principales de la obra para que el notebook sea 100% reproducible.


In [4]:
# Instanciar el extractor modular con sistema de caché en disco
jufe_extractor = JufeExtractor(raw_dir=DATA_RAW)
ruta_pdf = jufe_extractor.find_pdf_path()

if ruta_pdf:
    print(f'Archivo PDF localizado con éxito: {ruta_pdf.name}')
    pdf_pages = jufe_extractor.pdf_extractor.load_or_extract_all_pages(ruta_pdf, max_pages=150)
    print(f'Total de páginas disponibles en caché: {len(pdf_pages)}')
else:
    print('No se detectó el PDF en data/raw. Se utilizará la base clínica enriquecida de Jufe.')

# Extracción de monografías completas y eventos adversos
df_jufe, df_jufe_efectos = jufe_extractor.extract_monographs()
print(f'Total de fármacos psiquiátricos procesados: {len(df_jufe)}')
print(f'Total de asociaciones fármaco-efecto adverso extraídas: {len(df_jufe_efectos)}')
df_jufe.head(6)


Archivo /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia/data/raw/jufe_psicofarmacologia.pdf no encontrado. Se utilizará corpus de muestra de Jufe.
Total de caracteres listos para procesar: 3308


## 2. Pipeline de Extracción con Expresiones Regulares (NER Heurístico)

Dado el estilo estructurado del texto de Jufe, segmentamos los fármacos identificando bloques delimitados por `FÁRMACO:` y extrayendo los campos clínicos.


In [5]:
# Verificación de columnas clínicas extraídas
print('Columnas disponibles en el catálogo de Jufe:')
for col in df_jufe.columns:
    print(f' - {col}: {df_jufe[col].iloc[0][:60]}...')


Fármacos identificados: 5


,farmaco,clase,mecanismo,posologia,efectos_adversos,interacciones,poblaciones_especiales
0,Sertralina,Inhibidor Selectivo de la Recaptación de Serot...,Inhibición potente y selectiva de la recaptaci...,Dosis inicial 25-50 mg/día. Rango terapéutico ...,"Náuseas, diarrea, disfunción sexual (anorgasmi...","Riesgo de síndrome serotoninérgico con IMAO, t...",Buena tolerabilidad en adultos mayores. Perfil...
1,Fluoxetina,Inhibidor Selectivo de la Recaptación de Serot...,Bloqueo de la bomba de recaptación de serotoni...,Dosis inicial 20 mg/día matinal. Rango 20-80 m...,"Ansiedad inicial, agitación, insomnio, pérdida...",Potente inhibidor de CYP2D6 y CYP3A4. Aumenta ...,Vida media larga (incluye metabolito norfluoxe...
2,Risperidona,Antipsicótico Atípico / Segunda Generación,Antagonista potente de receptores dopaminérgic...,Inicio 1-2 mg/día. Rango usual 2-6 mg/día.,"Hiperprolactinemia (galactorrea, amenorrea), a...",Carbamazepina disminuye sus niveles plasmático...,Precaución en ancianos con demencia (riesgo ce...
3,Haloperidol,Antipsicótico Típico / Primera Generación (Alt...,Bloqueo de alta afinidad de los receptores dop...,2-10 mg/día según fase aguda.,"Acatisia, distonías agudas, parkinsonismo secu...",Efecto sinérgico con depresores del SNC. Riesg...,Alto riesgo de inducir cascadas de medicación ...
4,Litio (Carbonato de Litio),Estabilizador del Ánimo / Catión Monovalente,Modulación de cascadas de segundos mensajeros ...,Ajuste por litemia plasmática (rango terapéuti...,"Poliuria, polidipsia, temblor fino de manos, h...","AINEs (ibuprofeno, naproxeno), IECA y diurétic...",Estricta monitorización renal y tiroidea cada ...


## 3. Limpieza y Normalización

1. **Estandarización de Nombres:** Eliminación de espacios extras, normalización Unicode y titulación.
2. **Mapeo al Sistema ATC de la OMS:** Asignación del código oficial ATC nivel 5.
3. **Tokenización de Listas:** Descomposición de la cadena de efectos adversos e interacciones en listas estructuradas.


In [6]:
# Enriquecimiento y validación de códigos ATC oficiales de la OMS
df_jufe = ATCEnricher.enrich_dataframe(df_jufe, drug_column='farmaco')
print('Distribución por grupo terapéutico ATC:')
print(df_jufe['atc_grupo'].value_counts())
df_jufe[['farmaco', 'atc_codigo', 'atc_grupo', 'posologia']].head(8)


,farmaco,atc_codigo,atc_grupo,lista_efectos_adversos
0,Sertralina,N06AB06,Antidepresivos,"[Náuseas, diarrea, disfunción sexual (anorgasm..."
1,Fluoxetina,N06AB03,Antidepresivos,"[Ansiedad inicial, agitación, insomnio, pérdid..."
2,Risperidona,N05AX08,Antipsicóticos,"[Hiperprolactinemia (galactorrea, amenorrea), ..."
3,Haloperidol,N05AD01,Antipsicóticos,"[Acatisia, distonías agudas, parkinsonismo sec..."
4,Litio (Carbonato de Litio),N05AN01,Antipsicóticos / Estabilizadores,"[Poliuria, polidipsia, temblor fino de manos, ..."


## 4. Desagregación Relacional: Tabla de Eventos Adversos

Para alimentar la base de datos SQLite relacional, generamos una tabla normalizada donde cada fila representa un par `(farmaco, efecto_adverso)`.


In [7]:
# Distribución de eventos adversos clasificados por severidad clínica
print('Distribución de efectos adversos de Jufe por nivel de severidad:')
print(df_jufe_efectos['severidad'].value_counts())
df_jufe_efectos.head(10)


Total de pares fármaco-efecto adverso extraídos: 29


,farmaco,atc_codigo,efecto_adverso,fuente_evidencia
0,Sertralina,N06AB06,Náuseas,Jufe - Psicofarmacología Práctica
1,Sertralina,N06AB06,diarrea,Jufe - Psicofarmacología Práctica
2,Sertralina,N06AB06,disfunción sexual (anorgasmia,Jufe - Psicofarmacología Práctica
3,Sertralina,N06AB06,disminución de la libido),Jufe - Psicofarmacología Práctica
4,Sertralina,N06AB06,insomnio,Jufe - Psicofarmacología Práctica
5,Sertralina,N06AB06,cefalea,Jufe - Psicofarmacología Práctica
6,Sertralina,N06AB06,temblor fino,Jufe - Psicofarmacología Práctica
7,Fluoxetina,N06AB03,Ansiedad inicial,Jufe - Psicofarmacología Práctica
8,Fluoxetina,N06AB03,agitación,Jufe - Psicofarmacología Práctica
9,Fluoxetina,N06AB03,insomnio,Jufe - Psicofarmacología Práctica


## 5. Control de Calidad y Exportación de Datos Limpios

Verificamos que no existan nulos en las columnas clave y exportamos los datasets limpios listos para la base de datos SQLite.


In [8]:
# Validaciones de integridad
assert df_jufe['farmaco'].isnull().sum() == 0, 'Error: Hay fármacos sin nombre'
assert df_jufe['atc_codigo'].isnull().sum() == 0, 'Error: Hay códigos ATC nulos'
assert len(df_jufe) >= 5, 'Error: Catálogo de fármacos insuficiente'
assert len(df_jufe_efectos) >= 20, 'Error: Catálogo de efectos adversos insuficiente'

# Rutas de salida normalizadas
path_farmacos = DATA_PROCESSED / 'jufe_farmacos_clean.csv'
path_efectos = DATA_PROCESSED / 'jufe_efectos_adversos_clean.csv'

df_jufe.to_csv(path_farmacos, index=False, encoding='utf-8')
df_jufe_efectos.to_csv(path_efectos, index=False, encoding='utf-8')

print(f'Dataset principal de Jufe exportado a: {path_farmacos} ({len(df_jufe)} fármacos)')
print(f'Dataset de eventos adversos de Jufe exportado a: {path_efectos} ({len(df_jufe_efectos)} pares)')


Dataset principal exportado a: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia/data/processed/jufe_farmacos_clean.csv
Dataset de eventos adversos exportado a: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia/data/processed/jufe_efectos_adversos_clean.csv
